<h1> Converting Quantum ESPRESSO PWSCF SCF data to DeepMD-kit Format </h1>

In [ ]:
import dpdata
from pathlib import Path

[Documentação DeePMD-kit](https://docs.deepmodeling.com/projects/deepmd/en/stable/data/system.html)

DeePMD-kit aceita um `system` como estrutura de dados. Um *snapshot* é chamado de `frame`.

Um `system` pode conter muitos `frames` com mesmo número de átomos e tipos, ex: Zn $_2$ O $_2$.

Para conter dados com fórmulas diferentes (Zn $_4$ O $_4$, i.e, supercélulas maiores), geralmente é necessário dividir os dados em múltiplos sistemas, o que às vezes pode resultar em sistemas de frames esparsos.

In [ ]:
def get_pairs(inputs_dir, outputs_dir) -> list[list[str, str]]:
    in_files = sorted(inputs_dir.glob("ZnO*.in"))
    out_files = sorted(outputs_dir.glob("ZnO*.out"))

    pairs = [[str(in_f), str(out_f)] for in_f, out_f in zip(in_files, out_files)]

    return pairs


LDA_000_INPUTS = Path("/home/jvc/ZnO_database/scripts/LDA_000_INPUTS")
LDA_000_OUTPUTS = Path("/home/jvc/ZnO_database/scripts/LDA_000_OUTPUTS")

LDA_004_INPUTS = Path("/home/jvc/ZnO_database/scripts/LDA_004_INPUTS")
LDA_004_OUTPUTS = Path("/home/jvc/ZnO_database/scripts/LDA_004_OUTPUTS")

dataset = {
    "000": (LDA_000_INPUTS, LDA_000_OUTPUTS),
    "004": (LDA_004_INPUTS, LDA_004_OUTPUTS)

}
pairs_000 = get_pairs(*dataset["000"])
pairs_004 = get_pairs(*dataset["004"])
all_pairs = pairs_000 + pairs_004

multi_system = dpdata.MultiSystems()

for p in all_pairs:
    try:
        s = dpdata.LabeledSystem(p, fmt="qe/pw/scf")
        multi_system.append(s)
    except Exception as e:
        print(f"ERRO:{e}")
        print(f"IO: {p}\n")

print(multi_system)    


Como as minhas estruturas tem número de átomos diferentes (como vi no print: 4, 8, 12 átomos), o `LabeledSystem` pode reclamar se você tentar misturá-las em um único objeto. 

Nesse caso, a abordagem correta para o **DeepMD** é usar o `MultiSystems`:


In [53]:
print(f"Criando dataset: {multi_system}")
multi_system.to('deepmd/npy', 'LDA_deepmd_dataset', set_size=int(len(all_pairs) / 5))


print("Conversão concluída com sucesso!")

Criando dataset: MultiSystems (7 systems containing 3139 frames)
Conversão concluída com sucesso!


São criados 8 sistemas pois o dpdata `agrupa por número de átomos`.

Os dados no formato do DeePMD-kit está armazenado no diretório `LDA_deepmd_dataset`

---